This uses a pymc environment, not the workspace environment.





In [1]:
import pymc as pm
import numpy as np
import arviz as az
import matplotlib.pyplot as plt

In [3]:
rng = np.random.default_rng(2)
N = 2000

rng = np.random.default_rng(42)
N = 3000

# True params
beta_xc_true = 1.0     # x -> c
beta_yc_true = 1.0     # y -> c
sigma_c_true = 1.0
sigma_y_true = 1.0

# Exogenous x, independent y; c is a collider of x and y
x = rng.normal(0, 1, N)
y = rng.normal(0, sigma_y_true, N)                          # independent of x
c = beta_xc_true * x + beta_yc_true * y + rng.normal(0, sigma_c_true, N)


Note that y is completely independent of x in this model, but if we condition on c, we will see a spurious association between x and y. To avoid that we can use a full Bayesian model:

In [9]:

with pm.Model() as m_collider:
    # Priors
    beta_xc = pm.Normal("beta_xc", 0, 1)
    beta_yc = pm.Normal("beta_yc", 0, 1)
    sigma_c = pm.HalfNormal("sigma_c", 1)
    sigma_y = pm.HalfNormal("sigma_y", 1)
    beta_xy = pm.Normal("beta_xy", 0, 1)   # should be near zero
    intercept_y = pm.Normal("intercept_y", 0, 1)

    # y is independent of x; c is caused by x and y
    # but still we could measure y depends on c
    mu_y = beta_xy * x + intercept_y
    y_rv = pm.Normal("y", mu=mu_y, sigma=sigma_y, shape=N, observed=y)
    mu_c = beta_xc * x + beta_yc * y_rv
    c_rv = pm.Normal("c", mu=mu_c, sigma=sigma_c, shape=N, observed=c)

with m_collider:
    idata_collider = pm.sample()

az.summary(idata_collider)




Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_xc, beta_yc, sigma_c, sigma_y, beta_xy, intercept_y]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 1 seconds.


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta_xc,0.997,0.018,0.965,1.034,0.0,0.0,5899.0,3068.0,1.0
beta_yc,0.998,0.019,0.961,1.031,0.0,0.0,6364.0,3287.0,1.0
beta_xy,-0.006,0.018,-0.040,0.026,0.0,0.0,6344.0,3257.0,1.0
intercept_y,0.009,0.018,-0.028,0.042,0.0,0.0,7430.0,3686.0,1.0
sigma_c,1.003,0.013,0.980,1.030,0.0,0.0,5218.0,3163.0,1.0
sigma_y,0.998,0.013,0.975,1.024,0.0,0.0,6651.0,3243.0,1.0


Incorrect model (conditioning on collider):

In [8]:
with pm.Model() as m_B:
    alpha = pm.Normal("alpha", 0, 2)
    b_x   = pm.Normal("b_x", 0, 1)    
    b_c   = pm.Normal("b_c", 0, 1)
    sigma = pm.HalfNormal("sigma", 1)
    mu    = alpha + b_x * x + b_c * c
    y_lin = pm.Normal("y", mu=mu, sigma=sigma, shape=N, observed=y)
    idata_B = pm.sample()

az.summary(idata_B)

Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [alpha, b_x, b_c, sigma]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 1 seconds.


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
alpha,0.006,0.013,-0.020,0.030,0.0,0.0,4734.0,3342.0,1.0
b_x,-0.500,0.016,-0.527,-0.470,0.0,0.0,2895.0,2940.0,1.0
b_c,0.498,0.009,0.480,0.514,0.0,0.0,2916.0,3365.0,1.0
sigma,0.709,0.009,0.692,0.725,0.0,0.0,4443.0,3045.0,1.0
